# Tienda simulada con SQLite

Gestiona productos, precios, stock, carritos, cotizaciones y pedidos. La confirmación revalida las condiciones dentro de una transacción y evita pedidos duplicados.

In [11]:
#%pip -q install -r requirements-lab4-modular.txt

In [12]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
import pandas as pd
from lab4_asistente import config as C
from lab4_asistente.tienda import Tienda


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Crear y restaurar la tienda

In [13]:
catalogo = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
C.RUTA_TIENDA.mkdir(parents=True, exist_ok=True)
BASE = C.RUTA_TIENDA / "tienda_base.sqlite"
TRABAJO = C.RUTA_TIENDA / "tienda.sqlite"

Tienda.crear_base(BASE, catalogo.product_id, C.SEMILLA, *C.PRECIO_LOGNORMAL,
                  C.PRECIO_RANGO, C.PROB_AGOTADO, C.STOCK_MAX)
tienda = Tienda(BASE, TRABAJO, C.CANTIDAD_MAX)
tienda.restaurar()
print("Huella inicial:", tienda.huella())
print("Productos:", len(catalogo))


Huella inicial: afdd683c93168902
Productos: 10688


## Probar carrito, cotización y confirmación

In [14]:
# Escoge automáticamente un producto con stock para que la demostración sea ejecutable.
producto_id = next(p for p in catalogo.product_id if tienda.producto(p)["stock"] > 0)
carrito_id = tienda.nuevo_carrito()
carrito_demo = tienda.fijar_cantidad(carrito_id, producto_id, 1)
cot = tienda.cotizar(carrito_id, presupuesto_cent=100_000)
print("Cotización:", {k: cot[k] for k in ("cotizacion_id", "total_cent", "valida", "problemas")})
if cot["valida"]:
    pedido = tienda.confirmar(cot["cotizacion_id"], presupuesto_cent=100_000)
    repetido = tienda.confirmar(cot["cotizacion_id"], presupuesto_cent=100_000)
    print("Primera confirmación:", pedido["estado"])
    print("Repetición idempotente:", repetido["estado"], "pedido", repetido.get("pedido_id"))
print("Estado final:", tienda.resumen()["stock_negativo"], "stocks negativos")


Cotización: {'cotizacion_id': 'COT-1-1', 'total_cent': 1099, 'valida': True, 'problemas': []}
Primera confirmación: registrado
Repetición idempotente: ya_registrado pedido 1
Estado final: 0 stocks negativos


### Confirmación segura

La tienda revalida las condiciones y registra el pedido en una transacción. Si una comprobación falla, revierte los cambios. Una clave única por cotización evita pedidos duplicados.

## Batería de pruebas de la tienda

Cada prueba restaura la base para empezar desde el mismo inventario. Se muestran las acciones, el resultado esperado y el resultado real: consultar productos, editar el carrito, cotizar, confirmar, repetir una confirmación y rechazar operaciones inválidas.

In [15]:
def ejecutar_prueba_tienda(nombre, acciones, esperado, funcion_prueba):
    resultado = funcion_prueba()
    ok = resultado["ok"]
    print(f"\nPrueba: {nombre}")
    print("Acciones:", acciones)
    print("Esperado:", esperado)
    print("Resultado real:", resultado["detalle"])
    print("Estado:", "OK" if ok else "REVISAR")
    return {"Prueba": nombre, "Acciones": acciones, "Esperado": esperado,
            "Resultado real": resultado["detalle"], "Estado": "OK" if ok else "REVISAR"}

def producto_con_stock():
    return next(p for p in catalogo.product_id if tienda.producto(p)["stock"] > 0)

def _prueba_consulta_tienda():
    tienda.restaurar()
    pid = producto_con_stock()
    p = tienda.producto(pid)
    fila = catalogo.loc[catalogo.product_id == pid].iloc[0]
    return {"ok": p is not None and p["stock"] > 0,
            "detalle": f"{fila.titulo} | precio: {p['precio_cent']/100:.2f} | stock: {p['stock']}"}

def _prueba_carrito_tienda():
    tienda.restaurar()
    pid = producto_con_stock(); cid = tienda.nuevo_carrito()
    tienda.fijar_cantidad(cid, pid, 2)
    tienda.fijar_cantidad(cid, pid, 1)
    car = tienda.carrito(cid)
    return {"ok": len(car["items"]) == 1 and car["items"][0]["cantidad"] == 1,
            "detalle": f"carrito {cid}: {car['items']} | total: {car['total_cent']/100:.2f}"}

def _prueba_cotizacion_rechazada_tienda():
    tienda.restaurar()
    pid = producto_con_stock(); cid = tienda.nuevo_carrito()
    tienda.fijar_cantidad(cid, pid, 1)
    cot = tienda.cotizar(cid, presupuesto_cent=1)
    return {"ok": not cot["valida"] and any("presupuesto" in x for x in cot["problemas"]),
            "detalle": f"válida={cot['valida']} | problemas={cot['problemas']}"}

def _prueba_confirmacion_idempotente_tienda():
    tienda.restaurar()
    pid = producto_con_stock(); cid = tienda.nuevo_carrito()
    tienda.fijar_cantidad(cid, pid, 1)
    cot = tienda.cotizar(cid, presupuesto_cent=100_000)
    primero = tienda.confirmar(cot["cotizacion_id"], presupuesto_cent=100_000)
    segundo = tienda.confirmar(cot["cotizacion_id"], presupuesto_cent=100_000)
    resumen = tienda.resumen()
    return {"ok": primero["estado"] == "registrado" and segundo["estado"] == "ya_registrado" and len(resumen["pedidos"]) == 1,
            "detalle": f"primera={primero['estado']} | repetida={segundo['estado']} | pedidos={len(resumen['pedidos'])}"}

def _prueba_cotizacion_vencida_tienda():
    tienda.restaurar()
    pid = producto_con_stock(); cid = tienda.nuevo_carrito()
    tienda.fijar_cantidad(cid, pid, 1)
    cot = tienda.cotizar(cid, presupuesto_cent=100_000)
    tienda.fijar_cantidad(cid, pid, 2)
    pedido = tienda.confirmar(cot["cotizacion_id"], presupuesto_cent=100_000)
    return {"ok": pedido["estado"] == "rechazado" and any("cambio despues" in x for x in pedido["problemas"]),
            "detalle": f"estado={pedido['estado']} | problemas={pedido.get('problemas', [])}"}

casos_tienda = [
    ("Consulta de precio y stock", "buscar un producto disponible y leer su precio y existencias", "producto encontrado con precio y stock mayor que cero", _prueba_consulta_tienda),
    ("Edición del carrito", "añadir 2 unidades y luego fijar la cantidad en 1", "queda una línea con cantidad 1 y total actualizado", _prueba_carrito_tienda),
    ("Cotización sobre presupuesto", "cotizar una unidad con presupuesto de 0.01", "cotización inválida con motivo de presupuesto", _prueba_cotizacion_rechazada_tienda),
    ("Confirmación repetida", "confirmar una cotización válida dos veces", "primer estado registrado, segundo ya_registrado y un pedido", _prueba_confirmacion_idempotente_tienda),
    ("Cotización desactualizada", "cambiar el carrito después de cotizar y confirmar", "pedido rechazado porque cambió el carrito", _prueba_cotizacion_vencida_tienda),
]

filas_tienda = []
for nombre, acciones, esperado, funcion in casos_tienda:
    filas_tienda.append(ejecutar_prueba_tienda(nombre, acciones, esperado, funcion))

tabla_pruebas_tienda = pd.DataFrame(filas_tienda)
display(tabla_pruebas_tienda)


Prueba: Consulta de precio y stock
Acciones: buscar un producto disponible y leer su precio y existencias
Esperado: producto encontrado con precio y stock mayor que cero
Resultado real: The Thief: 1 (Queen's Thief) | precio: 10.99 | stock: 11
Estado: OK

Prueba: Edición del carrito
Acciones: añadir 2 unidades y luego fijar la cantidad en 1
Esperado: queda una línea con cantidad 1 y total actualizado
Resultado real: carrito 1: [{'product_id': '0060824972', 'cantidad': 1, 'precio_cent': 1099, 'stock': 11, 'subtotal_cent': 1099}] | total: 10.99
Estado: OK

Prueba: Cotización sobre presupuesto
Acciones: cotizar una unidad con presupuesto de 0.01
Esperado: cotización inválida con motivo de presupuesto
Resultado real: válida=False | problemas=['el total 10.99 supera el presupuesto 0.01']
Estado: OK

Prueba: Confirmación repetida
Acciones: confirmar una cotización válida dos veces
Esperado: primer estado registrado, segundo ya_registrado y un pedido
Resultado real: primera=registrado | repet

,Prueba,Acciones,Esperado,Resultado real,Estado
0,Consulta de precio y stock,buscar un producto disponible y leer su precio...,producto encontrado con precio y stock mayor q...,The Thief: 1 (Queen's Thief) | precio: 10.99 |...,OK
1,Edición del carrito,añadir 2 unidades y luego fijar la cantidad en 1,queda una línea con cantidad 1 y total actuali...,"carrito 1: [{'product_id': '0060824972', 'cant...",OK
2,Cotización sobre presupuesto,cotizar una unidad con presupuesto de 0.01,cotización inválida con motivo de presupuesto,válida=False | problemas=['el total 10.99 supe...,OK
3,Confirmación repetida,confirmar una cotización válida dos veces,"primer estado registrado, segundo ya_registrad...",primera=registrado | repetida=ya_registrado | ...,OK
4,Cotización desactualizada,cambiar el carrito después de cotizar y confirmar,pedido rechazado porque cambió el carrito,estado=rechazado | problemas=['el carrito camb...,OK
